In [1]:
import os
import sys
import numpy as np
import pandas as pd
import networkx as nx
import plotly.express as px
from pathlib import Path

# Make sure the GlobalModel utils and graph_plot are importable
NOTEBOOK_DIR = Path(os.getcwd())
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from utils import compute_distances_allvsall
from graph_plot import plot_networkx_plotly


In [ ]:
# ── Config ─────────────────────────────────────────────────────────────────
DATA_PATH  = str(NOTEBOOK_DIR / '../../dataset/data_smooth_erratic.feather')
DATE_COL   = 'date'
TARGET_COL = 'value'

# Distance metric to use — pick one:
# 'euclidean' | 'manhattan' | 'cid' | 'dtw' | 'hamming' | 'amplitude_offset'
# 'slope_consistency' | 'lorentzian' | 'sbd' | 'msm' | 'edr' | 'lcss'
# 'twed' | 'erp' | 'stid' | 'phase_invariance'
METRIC = 'cid'

# DISTANCE_THRESHOLD is set AFTER inspecting the distribution in the next cells.
# Lower = stricter (fewer edges). Set to None here; assign after seeing the histogram.
DISTANCE_THRESHOLD = 70



In [3]:
val_size = 30
forecast_horizon = 153
train_size = 761 - val_size - forecast_horizon  # 455
lookback_window = 30

In [4]:
# ── Load dataset & build wide pivot (item_id × date) ──────────────────────
df = pd.read_feather(DATA_PATH)
df[DATE_COL] = pd.to_datetime(df[DATE_COL])
df = df.sort_values([DATE_COL, 'item_id']).reset_index(drop=True)

df_wide = (
    df.pivot_table(index='item_id', columns=DATE_COL, values=TARGET_COL, aggfunc='sum')
    .fillna(0)
)

# Restrict to the training window only — distances must not see val/test data
df_wide = df_wide.iloc[:, :train_size]

# Optional category labels for colouring nodes in the plot
cat_labels_dict = (
    df.drop_duplicates('item_id').set_index('item_id')['cat_label'].to_dict()
    if 'cat_label' in df.columns else {}
)

item_ids = df_wide.index.tolist()
print(f"Loaded: {len(item_ids)} products × {df_wide.shape[1]} time steps (train only)")


Loaded: 972 products × 578 time steps (train only)


In [5]:
# ── Compute all-vs-all distance matrix (GPU-accelerated if available) ──────
all_ts = df_wide.values.astype(np.float32)   # (N, T)

print(f"Computing {METRIC} distances for {len(item_ids)} × {len(item_ids)} pairs...")
dist_matrix = compute_distances_allvsall(all_ts, metric=METRIC)

print(f"Done.  Matrix shape: {dist_matrix.shape}")
print(f"Value range (excl. diagonal): [{np.triu(dist_matrix, k=1)[np.triu(dist_matrix, k=1) > 0].min():.4f}, "
      f"{dist_matrix.max():.4f}]")


Computing cid distances for 972 × 972 pairs...
Done.  Matrix shape: (972, 972)
Value range (excl. diagonal): [56.5027, 153055.5312]


In [6]:
# ── Apply threshold → keep only connected products ─────────────────────────
N = len(item_ids)

# Vectorised: keep upper-triangle pairs where distance <= threshold
mask = np.triu(dist_matrix <= DISTANCE_THRESHOLD, k=1)  # exclude self-loops
rows, cols = np.where(mask)

G_full = nx.Graph()
G_full.add_nodes_from(item_ids)
for i, j in zip(rows, cols):
    G_full.add_edge(item_ids[i], item_ids[j], weight=float(dist_matrix[i, j]))

# Sub-graph: only nodes with at least one edge
connected_nodes = [n for n, d in G_full.degree() if d > 0]
G = G_full.subgraph(connected_nodes).copy()

# Attach category label to each node (used by plot_networkx_plotly for colouring)
for node in G.nodes():
    G.nodes[node]['cat_label'] = cat_labels_dict.get(node, 'Unknown')

isolated = N - G.number_of_nodes()
print(f"Distance threshold = {DISTANCE_THRESHOLD:.4f}  |  metric = {METRIC}")
print(f"  Connected products : {G.number_of_nodes():>5d}  ({isolated} isolated, removed)")
print(f"  Edges              : {G.number_of_edges():>5d}")


Distance threshold = 70.0000  |  metric = cid
  Connected products :    33  (939 isolated, removed)
  Edges              :   116


In [7]:
# ── Interactive plot (only connected products are shown) ───────────────────
plot_networkx_plotly(
    G,
    title=(
        f'All-vs-All Product Distance  ({METRIC} ≤ {DISTANCE_THRESHOLD:.4f})'
        f'  —  {G.number_of_nodes()} connected products  |  {G.number_of_edges()} edges'
    ),
)
